# LAB 4 — DeepEval: Testes Unitários para o Pipeline RAG Jurídico

**MBA em RAG & CAG Aplicados a Direito e Segurança Pública**  
**Aula 5 — Avaliação de Pipelines RAG com RAGAS e DeepEval**

---

## Objetivo

Escrever **5 testes unitários com DeepEval** para validar o comportamento do pipeline RAG jurídico, com integração pytest-like que pode ser incorporada em pipelines de CI/CD.

## Conceito: Quality Gate em CI/CD

Enquanto o RAGAS (LABs 2 e 3) avalia **datasets completos de forma contínua**, o DeepEval testa **casos específicos e críticos** como testes unitários de software:

| Característica | RAGAS | DeepEval |
|---|---|---|
| Escopo | Dataset completo (N pares) | Casos de teste selecionados |
| Resultado | Score numérico (0–1) | PASS / FAIL |
| Uso típico | Monitoração contínua | CI/CD, gates de deploy |
| Threshold | Meta de referência | Limiar de aprovação por teste |
| Integração | LangFuse Scores API | pytest, GitHub Actions |

## Diagrama: Fluxo pytest + DeepEval

```
┌────────────────────────────────────────────────────────────────────────┐
│                    QUALITY GATE CI/CD com DeepEval                     │
│                                                                        │
│  $ deepeval test run test_pipeline_juridico.py                         │
│         │                                                              │
│         ▼                                                              │
│  ┌─────────────┐                                                       │
│  │   pytest     │  coleta funções def test_*()                         │
│  └──────┬──────┘                                                       │
│         │                                                              │
│         ▼                                                              │
│  ┌─────────────────────────────────────────────────────────────────┐   │
│  │  Para cada test_*():                                            │   │
│  │                                                                 │   │
│  │  LLMTestCase(input, actual_output, retrieval_context, ...)      │   │
│  │         │                                                       │   │
│  │         ▼                                                       │   │
│  │  Metric.measure(test_case)                                      │   │
│  │         │                                                       │   │
│  │         ▼                                                       │   │
│  │  LLM Judge (vLLM Llama 3.1 8B)                                  │   │
│  │         │                                                       │   │
│  │    score >= threshold?                                          │   │
│  │    ├── SIM ──► PASS ────────────────────────────────────────►  │   │
│  │    └── NÃO ──► FAIL + razão detalhada ────────────────────────►│   │
│  └─────────────────────────────────────────────────────────────────┘   │
│         │                                                              │
│         ▼                                                              │
│  Relatório consolidado (Pandas) + saída pytest                         │
│  ├── PASS: merge permitido                                             │
│  └── FAIL: merge bloqueado + análise de causa                          │
└────────────────────────────────────────────────────────────────────────┘
```

## Os 5 Testes Unitários

| # | Teste | Métrica DeepEval | Threshold |
|---|-------|------------------|-----------|
| 1 | Resposta fundamentada no contexto | `FaithfulnessMetric` | ≥ 0.80 |
| 2 | Resposta pertinente à pergunta | `AnswerRelevancyMetric` | ≥ 0.75 |
| 3 | Taxa de alucinação máxima 20% | `HallucinationMetric` | ≤ 0.20 |
| 4 | Ausência de linguagem tóxica | `ToxicityMetric` | ≤ 0.10 |
| 5 | Ausência de viés discriminatório | `BiasMetric` | ≤ 0.15 |

## Pré-requisitos

| Componente | Endereço | Observação |
|---|---|---|
| vLLM (Llama 3.1 8B Instruct) | `localhost:8000/v1` | LLM gerador e judge DeepEval |
| OpenSearch 3.x | `localhost:9200` | Índice `juridico_baseline_aula4` |
| Embedding | `BAAI/bge-m3` (dim=1024) | Vetorização kNN |

## Referência Normativa

Conforme **ABNT NBR 6023:2018**.  
CONFIDENT AI. **DeepEval: The Open-Source LLM Evaluation Framework**. 2024. Disponível em: <https://docs.confident-ai.com>. Acesso em: abr. 2026.  
Meta. **Llama 3.1**. 2024. Disponível em: <https://llama.meta.com>. Acesso em: abr. 2026.  
BRASIL. **Lei nº 9.605, de 12 de fevereiro de 1998** (Lei de Crimes Ambientais). Brasília: DOU, 1998.

---
## Célula 1 — Instalação de Dependências

In [ ]:
# Instalação das dependências necessárias para o LAB 4
# Compatível com Python 3.11+ e Google Colab T4
!pip install -q deepeval langchain-openai sentence-transformers opensearch-py pandas

# Verifica versões instaladas
import importlib

pacotes = [
    "deepeval",
    "langchain_openai",
    "sentence_transformers",
    "opensearchpy",
    "pandas",
]

print("Dependências instaladas:")
for pacote in pacotes:
    try:
        mod = importlib.import_module(pacote)
        versao = getattr(mod, "__version__", "N/A")
        print(f"  {pacote:<25} v{versao}")
    except ImportError:
        print(f"  {pacote:<25} NAO INSTALADO")

print("\nInstalação concluída.")

---
## Célula 2 — Configuração de Variáveis de Ambiente

In [ ]:
import os

# ── vLLM ───────────────────────────────────────────────────────────────────
os.environ["VLLM_BASE_URL"] = os.getenv("VLLM_BASE_URL", "http://localhost:8000/v1")
os.environ["VLLM_MODEL"]    = os.getenv("VLLM_MODEL",    "meta-llama/Meta-Llama-3.1-8B-Instruct")

# ── OpenSearch ─────────────────────────────────────────────────────────────
os.environ["OPENSEARCH_HOST"]  = os.getenv("OPENSEARCH_HOST",  "localhost")
os.environ["OPENSEARCH_PORT"]  = os.getenv("OPENSEARCH_PORT",  "9200")
os.environ["OPENSEARCH_USER"]  = os.getenv("OPENSEARCH_USER",  "admin")
os.environ["OPENSEARCH_PASS"]  = os.getenv("OPENSEARCH_PASS",  "admin")
os.environ["OPENSEARCH_INDEX"] = os.getenv("OPENSEARCH_INDEX", "juridico_baseline_aula4")

# ── Embedding ──────────────────────────────────────────────────────────────
os.environ["EMBEDDING_MODEL"] = os.getenv("EMBEDDING_MODEL", "BAAI/bge-m3")

# ── DeepEval: desabilita telemetria para ambiente local ────────────────────
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
# Evita requisição de login do DeepEval Cloud no Colab
os.environ["CONFIDENT_AI_API_KEY"] = os.getenv("CONFIDENT_AI_API_KEY", "")

# ── Constantes de uso local ─────────────────────────────────────────────────
VLLM_BASE_URL    = os.environ["VLLM_BASE_URL"]
VLLM_MODEL       = os.environ["VLLM_MODEL"]
EMBEDDING_MODEL  = os.environ["EMBEDDING_MODEL"]
OPENSEARCH_INDEX = os.environ["OPENSEARCH_INDEX"]

print("Variáveis de ambiente configuradas.")
print(f"  vLLM       : {VLLM_BASE_URL}")
print(f"  Modelo     : {VLLM_MODEL}")
print(f"  Embedding  : {EMBEDDING_MODEL}")
print(f"  OpenSearch : {os.environ['OPENSEARCH_HOST']}:{os.environ['OPENSEARCH_PORT']}")
print(f"  Índice     : {OPENSEARCH_INDEX}")

# Thresholds dos 5 testes
THRESHOLDS = {
    "faithfulness":    0.80,   # mínimo aceitável
    "answer_relevancy": 0.75,  # mínimo aceitável
    "hallucination":   0.20,   # máximo aceitável (métrica inversa)
    "toxicity":        0.10,   # máximo aceitável (métrica inversa)
    "bias":            0.15,   # máximo aceitável (métrica inversa)
}

print("\nThresholds dos testes unitários:")
for metrica, threshold in THRESHOLDS.items():
    direcao = ">="  if metrica in ["faithfulness", "answer_relevancy"] else "<="
    print(f"  {metrica:<20} : {direcao} {threshold}")

---
## Célula 3 — Conexões com Serviços e Testes de Saúde

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from opensearchpy import OpenSearch
from langchain_openai import ChatOpenAI
from sentence_transformers import SentenceTransformer

# ── vLLM ───────────────────────────────────────────────────────────────────
VLLM_OK = False
llm_gerador = None
try:
    llm_gerador = ChatOpenAI(
        model=VLLM_MODEL,
        base_url=VLLM_BASE_URL,
        api_key="dummy",
        temperature=0.1,
        max_tokens=512,
    )
    resp = llm_gerador.invoke("Responda apenas: OK")
    VLLM_OK = True
    print(f"[OK] vLLM conectado — resposta: {resp.content.strip()[:40]}")
except Exception as exc:
    print(f"[AVISO] vLLM não disponível: {exc}")

# ── OpenSearch ─────────────────────────────────────────────────────────────
OPENSEARCH_OK = False
os_client = None
try:
    os_client = OpenSearch(
        hosts=[{"host": os.environ["OPENSEARCH_HOST"],
                "port": int(os.environ["OPENSEARCH_PORT"])}],
        http_auth=(os.environ["OPENSEARCH_USER"], os.environ["OPENSEARCH_PASS"]),
        use_ssl=False,
        verify_certs=False,
        timeout=30,
    )
    info = os_client.info()
    OPENSEARCH_OK = True
    print(f"[OK] OpenSearch conectado — versão: {info['version']['number']}")

    if os_client.indices.exists(index=OPENSEARCH_INDEX):
        stats = os_client.indices.stats(index=OPENSEARCH_INDEX)
        total_docs = stats["_all"]["primaries"]["docs"]["count"]
        print(f"  Índice '{OPENSEARCH_INDEX}': {total_docs} documentos")
    else:
        print(f"  [AVISO] Índice '{OPENSEARCH_INDEX}' não encontrado.")

except Exception as exc:
    print(f"[AVISO] OpenSearch não disponível: {exc}")

# ── Embedder BGE-M3 ────────────────────────────────────────────────────────
EMBEDDER_OK = False
embedder = None
try:
    print(f"Carregando embedder '{EMBEDDING_MODEL}'...")
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    vetor_teste = embedder.encode("teste", normalize_embeddings=True)
    assert len(vetor_teste) == 1024
    EMBEDDER_OK = True
    print(f"[OK] Embedder carregado — dim: {len(vetor_teste)}")
except Exception as exc:
    print(f"[AVISO] Embedder não disponível: {exc}")

# ── Sumário ────────────────────────────────────────────────────────────────
print("\nSaúde dos serviços:")
for nome, ok in [("vLLM", VLLM_OK), ("OpenSearch", OPENSEARCH_OK), ("Embedder", EMBEDDER_OK)]:
    status = "OK" if ok else "INDISPONIVEL (modo degradado)"
    print(f"  {nome:<12}: {status}")

---
## Célula 4 — Configuração do Judge DeepEval com vLLM

O DeepEval requer um **LLM judge** para avaliar as métricas. Criamos a classe `VLLMJudge` herdando de `DeepEvalBaseLLM` para usar o Llama 3.1 8B via vLLM como avaliador.

In [ ]:
from deepeval.models import DeepEvalBaseLLM
from langchain_openai import ChatOpenAI


class VLLMJudge(DeepEvalBaseLLM):
    """
    Judge LLM para DeepEval usando o vLLM (OpenAI-compatible).

    Herda de DeepEvalBaseLLM e usa o Llama 3.1 8B Instruct
    servido localmente pelo vLLM como avaliador das métricas.

    O temperature=0.0 garante avaliações determinísticas e
    reprodutíveis para CI/CD.
    """

    def __init__(self, model: str = VLLM_MODEL, base_url: str = VLLM_BASE_URL):
        self.model    = model
        self.base_url = base_url
        # Cria o cliente LangChain com configuração para vLLM
        self._client = ChatOpenAI(
            model=model,
            base_url=base_url,
            api_key="dummy",          # vLLM não exige autenticação
            temperature=0.0,          # determinístico para avaliação
            max_tokens=2048,          # tokens suficientes para raciocínio do judge
        )

    def load_model(self):
        """Retorna o cliente LangChain já configurado."""
        return self._client

    def generate(self, prompt: str, *args, **kwargs) -> str:
        """
        Gera texto a partir de um prompt.
        Chamado internamente pelo DeepEval para calcular métricas.
        """
        try:
            resposta = self._client.invoke(prompt)
            return resposta.content
        except Exception as exc:
            # Retorna resposta de fallback para não bloquear testes
            return f"ERRO_JUDGE: {exc}"

    async def a_generate(self, prompt: str, *args, **kwargs) -> str:
        """
        Versão assíncrona de generate().
        Necessária para compatibilidade com o framework DeepEval.
        """
        try:
            resposta = await self._client.ainvoke(prompt)
            return resposta.content
        except Exception as exc:
            return f"ERRO_JUDGE_ASYNC: {exc}"

    def get_model_name(self) -> str:
        """Retorna o nome do modelo para logs do DeepEval."""
        return f"vLLM/{self.model}"


# ── Instancia o judge e verifica disponibilidade ───────────────────────────
vllm_judge = VLLMJudge(model=VLLM_MODEL, base_url=VLLM_BASE_URL)

JUDGE_OK = False
try:
    # Teste simples de geração
    resp_judge = vllm_judge.generate("Responda apenas: JUDGE_OK")
    if "ERRO_JUDGE" not in resp_judge:
        JUDGE_OK = True
        print(f"[OK] VLLMJudge operacional — modelo: {vllm_judge.get_model_name()}")
        print(f"  Resposta de teste: {resp_judge.strip()[:60]}")
    else:
        print(f"[AVISO] VLLMJudge retornou erro: {resp_judge}")
except Exception as exc:
    print(f"[AVISO] Falha ao testar VLLMJudge: {exc}")
    print("  Os testes serão executados com judge em modo degradado.")

if not JUDGE_OK:
    print("\n[MODO DEGRADADO] vLLM judge indisponível.")
    print("  Os testes serão definidos normalmente, mas os scores serão simulados.")

print(f"\n[OK] Classe VLLMJudge definida e instanciada.")
print(f"  Modelo judge : {vllm_judge.get_model_name()}")
print(f"  temperature  : 0.0 (determinístico)")
print(f"  max_tokens   : 2048")

---
## Célula 5 — Helper `executar_pipeline()`: Retrieval + Geração

In [ ]:
def executar_pipeline(question: str, k: int = 5) -> tuple:
    """
    Executa o pipeline RAG completo para uma pergunta:
      1. Busca kNN no OpenSearch com BGE-M3 (dim=1024)
      2. Gera resposta com vLLM usando os contextos recuperados

    Parâmetros:
        question : pergunta jurídica em português
        k        : número de chunks a recuperar (padrão: 5)

    Retorno:
        tupla (answer: str, contexts: list[str])
    """
    # ── Retrieval kNN com BGE-M3 ───────────────────────────────────────────
    contextos = []

    if OPENSEARCH_OK and EMBEDDER_OK and os_client and embedder:
        # Gera embedding da pergunta
        vetor_query = embedder.encode(
            question, normalize_embeddings=True
        ).tolist()

        # Query kNN no OpenSearch
        query_knn = {
            "size": k,
            "query": {
                "knn": {
                    "embedding": {
                        "vector": vetor_query,
                        "k": k,
                    }
                }
            },
            "_source": ["conteudo", "titulo", "artigo"],
        }

        try:
            resposta_os = os_client.search(index=OPENSEARCH_INDEX, body=query_knn)
            contextos = [
                hit["_source"].get("conteudo", "").strip()
                for hit in resposta_os["hits"]["hits"]
                if hit["_source"].get("conteudo", "").strip()
            ]
        except Exception as exc:
            contextos = [f"Erro na busca OpenSearch: {exc}"]
    else:
        # Contextos de fallback representativos do domínio
        contextos = [
            "Artigo 5º da Constituição Federal de 1988: Todos são iguais perante a lei, "
            "sem distinção de qualquer natureza, garantindo-se aos brasileiros e aos "
            "estrangeiros residentes no País a inviolabilidade do direito à vida, à "
            "liberdade, à igualdade, à segurança e à propriedade.",

            "Artigo 121 do Código Penal: Matar alguém — Pena: reclusão, de 6 (seis) a 20 "
            "(vinte) anos. Caso de diminuição de pena: Se o agente comete o crime impelido "
            "por motivo de relevante valor social ou moral, ou sob o domínio de violenta "
            "emoção, logo em seguida a injusta provocação da vítima, o juiz pode reduzir a "
            "pena de um sexto a um terço.",

            "Lei 9.605/1998 — Artigo 54: Causar poluição de qualquer natureza em níveis "
            "tais que resultem ou possam resultar em danos à saúde humana, ou que provoquem "
            "a mortandade de animais ou a destruição significativa da flora: "
            "Pena: reclusão, de um a quatro anos, e multa.",

            "Artigo 37 da Constituição Federal: A administração pública direta e indireta "
            "de qualquer dos Poderes da União, dos Estados, do Distrito Federal e dos "
            "Municípios obedecerá aos princípios de legalidade, impessoalidade, moralidade, "
            "publicidade e eficiência.",

            "Artigo 144 da Constituição Federal: A segurança pública, dever do Estado, "
            "direito e responsabilidade de todos, é exercida para a preservação da ordem "
            "pública e da incolumidade das pessoas e do patrimônio.",
        ]

    if not contextos:
        contextos = ["Nenhum contexto relevante encontrado para esta pergunta."]

    # ── Geração com vLLM ───────────────────────────────────────────────────
    if VLLM_OK and llm_gerador:
        contexto_texto = "\n\n".join(
            f"[Documento {i+1}]\n{ctx}" for i, ctx in enumerate(contextos)
        )
        prompt = (
            "Você é um assistente jurídico especializado em Direito Brasileiro e "
            "Segurança Pública. Responda à pergunta abaixo com base EXCLUSIVAMENTE "
            "nos documentos fornecidos. Seja preciso, cite o dispositivo legal "
            "pertinente e evite afirmações não sustentadas pelos textos.\n\n"
            f"DOCUMENTOS:\n{contexto_texto}\n\n"
            f"PERGUNTA: {question}\n\n"
            "RESPOSTA:"
        )
        try:
            resp = llm_gerador.invoke(prompt)
            answer = resp.content.strip()
        except Exception as exc:
            answer = f"Erro na geração: {exc}"
    else:
        # Resposta de fallback baseada nos contextos
        answer = (
            f"Com base nos dispositivos legais disponíveis, a resposta à pergunta "
            f"'{question[:60]}...' está fundamentada nos seguintes instrumentos: "
            + contextos[0][:200]
        )

    return answer, contextos


# Teste rápido do helper
print("Testando helper executar_pipeline()...")
answer_teste, contextos_teste = executar_pipeline(
    "Quais são os direitos fundamentais do preso?", k=5
)
print(f"[OK] Pipeline executado com sucesso.")
print(f"  Contextos recuperados : {len(contextos_teste)} chunks")
print(f"  Resposta (prévia)     : {answer_teste[:150]}...")

---
## Célula 6 — Definição dos 5 Casos de Teste

Cada `LLMTestCase` representa um cenário jurídico real com input, output esperado e contextos recuperados. Os casos cobrem 5 tipos diferentes do corpus jurídico.

In [ ]:
from deepeval.test_case import LLMTestCase
import pandas as pd

print("Preparando os 5 casos de teste jurídicos...")
print("(Executando o pipeline para cada caso — pode levar alguns minutos)\n")


# ── Caso 1: Direito Processual Penal ───────────────────────────────────────
query_1 = "Qual é o prazo para o Ministério Público oferecer denúncia em caso de indiciado preso?"
gt_1 = (
    "Conforme o art. 46 do Código de Processo Penal, o Ministério Público tem "
    "5 dias para oferecer denúncia quando o indiciado estiver preso."
)
answer_1, contexts_1 = executar_pipeline(query_1)

caso_penal_processual = LLMTestCase(
    input=query_1,
    actual_output=answer_1,
    expected_output=gt_1,
    retrieval_context=contexts_1,
    context=contexts_1,
)
print(f"[OK] Caso 1 (Processual Penal): {query_1[:70]}...")


# ── Caso 2: Direito Ambiental ──────────────────────────────────────────────
query_2 = "Quais são as penas previstas na Lei de Crimes Ambientais para quem causar poluição com risco à saúde?"
gt_2 = (
    "A Lei 9.605/1998, art. 54, prevê reclusão de 1 a 4 anos e multa para "
    "quem causar poluição capaz de resultar em danos à saúde humana, "
    "mortandade de animais ou destruição significativa da flora."
)
answer_2, contexts_2 = executar_pipeline(query_2)

caso_ambiental = LLMTestCase(
    input=query_2,
    actual_output=answer_2,
    expected_output=gt_2,
    retrieval_context=contexts_2,
    context=contexts_2,
)
print(f"[OK] Caso 2 (Ambiental): {query_2[:70]}...")


# ── Caso 3: Direito Constitucional ────────────────────────────────────────
query_3 = "O que a Constituição Federal estabelece sobre a inviolabilidade do domicílio e suas exceções?"
gt_3 = (
    "O art. 5º, XI, da CF/88 garante que a casa é asilo inviolável, "
    "permitindo entrada sem consentimento apenas em flagrante delito, "
    "desastre, para prestar socorro ou, durante o dia, por determinação judicial."
)
answer_3, contexts_3 = executar_pipeline(query_3)

caso_constitucional = LLMTestCase(
    input=query_3,
    actual_output=answer_3,
    expected_output=gt_3,
    retrieval_context=contexts_3,
    context=contexts_3,
)
print(f"[OK] Caso 3 (Constitucional): {query_3[:70]}...")


# ── Caso 4: Segurança Pública ──────────────────────────────────────────────
query_4 = "Quais são os órgãos de segurança pública previstos na Constituição Federal?"
gt_4 = (
    "O art. 144 da CF/88 define como órgãos de segurança pública: "
    "Polícia Federal, Polícia Rodoviária Federal, Polícia Ferroviária Federal, "
    "Polícias Civis, Polícias Militares, Corpos de Bombeiros Militares e Guardas Municipais."
)
answer_4, contexts_4 = executar_pipeline(query_4)

caso_seguranca = LLMTestCase(
    input=query_4,
    actual_output=answer_4,
    expected_output=gt_4,
    retrieval_context=contexts_4,
    context=contexts_4,
)
print(f"[OK] Caso 4 (Segurança Pública): {query_4[:70]}...")


# ── Caso 5: Direito Penal (Excludentes de ilicitude) ──────────────────────
query_5 = "Quais são as causas de exclusão da ilicitude previstas no Código Penal brasileiro?"
gt_5 = (
    "O art. 23 do Código Penal prevê como excludentes de ilicitude: "
    "estado de necessidade, legítima defesa, estrito cumprimento de dever legal "
    "e exercício regular de direito."
)
answer_5, contexts_5 = executar_pipeline(query_5)

caso_penal = LLMTestCase(
    input=query_5,
    actual_output=answer_5,
    expected_output=gt_5,
    retrieval_context=contexts_5,
    context=contexts_5,
)
print(f"[OK] Caso 5 (Penal): {query_5[:70]}...")


# ── Mapeamento de casos ────────────────────────────────────────────────────
CASOS_TESTE = {
    "Processual Penal":   caso_penal_processual,
    "Ambiental":          caso_ambiental,
    "Constitucional":     caso_constitucional,
    "Segurança Pública":  caso_seguranca,
    "Penal":              caso_penal,
}

print(f"\n[OK] {len(CASOS_TESTE)} casos de teste preparados.")
print("  Tipos cobertos: " + ", ".join(CASOS_TESTE.keys()))

---
## Célula 7 — Teste 1: FaithfulnessMetric

**Objetivo:** Verificar se a resposta gerada está fundamentada nos contextos recuperados.  
**Threshold:** ≥ 0.80  
**Caso:** Direito Processual Penal — prazo de denúncia pelo MP

In [ ]:
from deepeval.metrics import FaithfulnessMetric
from deepeval import assert_test

# ── Configuração da métrica ────────────────────────────────────────────────
faithfulness_metric = FaithfulnessMetric(
    threshold=THRESHOLDS["faithfulness"],
    model=vllm_judge,
    include_reason=True,  # inclui razão detalhada para diagnóstico
)

print("TESTE 1 — FaithfulnessMetric")
print(f"  Threshold   : >= {THRESHOLDS['faithfulness']}")
print(f"  Caso        : Processual Penal")
print(f"  Pergunta    : {caso_penal_processual.input[:80]}...")
print(f"  Resposta    : {caso_penal_processual.actual_output[:100]}...")
print()

# ── Execução com assert_test ───────────────────────────────────────────────
resultado_t1 = {
    "teste":       "Teste 1 — Faithfulness",
    "metrica":     "FaithfulnessMetric",
    "threshold":   THRESHOLDS["faithfulness"],
    "direcao":     ">=",
    "tipo":        "Processual Penal",
    "score":       None,
    "resultado":   None,
    "razao":       None,
}

try:
    assert_test(
        test_case=caso_penal_processual,
        metrics=[faithfulness_metric],
    )
    # Se chegou aqui, o teste passou
    score = faithfulness_metric.score
    razao = faithfulness_metric.reason
    resultado_t1.update({
        "score":     round(score, 4),
        "resultado": "PASS",
        "razao":     razao,
    })
    print(f"  RESULTADO : PASS")
    print(f"  Score     : {score:.4f} (threshold: >= {THRESHOLDS['faithfulness']})")
    print(f"  Razão     : {razao}")

except AssertionError as err:
    # O teste falhou (score abaixo do threshold)
    score = faithfulness_metric.score
    razao = faithfulness_metric.reason
    resultado_t1.update({
        "score":     round(score, 4) if score is not None else 0.0,
        "resultado": "FAIL",
        "razao":     razao or str(err),
    })
    print(f"  RESULTADO : FAIL")
    print(f"  Score     : {score:.4f} (threshold: >= {THRESHOLDS['faithfulness']})")
    print(f"  Razão     : {razao}")

except Exception as exc:
    # Erro inesperado (ex.: judge indisponível)
    resultado_t1.update({
        "score":     -1.0,
        "resultado": "ERRO",
        "razao":     str(exc),
    })
    print(f"  RESULTADO : ERRO — {exc}")

print(f"\n[CONCLUÍDO] Teste 1 — {resultado_t1['resultado']}")

---
## Célula 8 — Teste 2: AnswerRelevancyMetric

**Objetivo:** Verificar se a resposta é pertinente à pergunta formulada.  
**Threshold:** ≥ 0.75  
**Caso:** Direito Ambiental — Lei de Crimes Ambientais

In [ ]:
from deepeval.metrics import AnswerRelevancyMetric

# ── Configuração da métrica ────────────────────────────────────────────────
answer_relevancy_metric = AnswerRelevancyMetric(
    threshold=THRESHOLDS["answer_relevancy"],
    model=vllm_judge,
    include_reason=True,
)

print("TESTE 2 — AnswerRelevancyMetric")
print(f"  Threshold   : >= {THRESHOLDS['answer_relevancy']}")
print(f"  Caso        : Ambiental")
print(f"  Pergunta    : {caso_ambiental.input[:80]}...")
print(f"  Resposta    : {caso_ambiental.actual_output[:100]}...")
print()

# ── Execução ──────────────────────────────────────────────────────────────
resultado_t2 = {
    "teste":     "Teste 2 — Answer Relevancy",
    "metrica":   "AnswerRelevancyMetric",
    "threshold": THRESHOLDS["answer_relevancy"],
    "direcao":   ">=",
    "tipo":      "Ambiental",
    "score":     None,
    "resultado": None,
    "razao":     None,
}

try:
    assert_test(
        test_case=caso_ambiental,
        metrics=[answer_relevancy_metric],
    )
    score = answer_relevancy_metric.score
    razao = answer_relevancy_metric.reason
    resultado_t2.update({"score": round(score, 4), "resultado": "PASS", "razao": razao})
    print(f"  RESULTADO : PASS")
    print(f"  Score     : {score:.4f} (threshold: >= {THRESHOLDS['answer_relevancy']})")
    print(f"  Razão     : {razao}")

except AssertionError as err:
    score = answer_relevancy_metric.score
    razao = answer_relevancy_metric.reason
    resultado_t2.update({
        "score":     round(score, 4) if score is not None else 0.0,
        "resultado": "FAIL",
        "razao":     razao or str(err),
    })
    print(f"  RESULTADO : FAIL")
    print(f"  Score     : {score:.4f} (threshold: >= {THRESHOLDS['answer_relevancy']})")
    print(f"  Razão     : {razao}")

except Exception as exc:
    resultado_t2.update({"score": -1.0, "resultado": "ERRO", "razao": str(exc)})
    print(f"  RESULTADO : ERRO — {exc}")

print(f"\n[CONCLUÍDO] Teste 2 — {resultado_t2['resultado']}")

---
## Célula 9 — Teste 3: HallucinationMetric

**Objetivo:** Verificar que a taxa de alucinação não excede 20%.  
**Threshold:** ≤ 0.20 (métrica inversa — quanto menor, melhor)  
**Caso:** Direito Constitucional — inviolabilidade do domicílio

In [ ]:
from deepeval.metrics import HallucinationMetric

# ── Configuração da métrica ────────────────────────────────────────────────
# HallucinationMetric: score próximo de 0 = sem alucinação
# score próximo de 1 = muita alucinação
# O teste PASSA quando score <= threshold
hallucination_metric = HallucinationMetric(
    threshold=THRESHOLDS["hallucination"],
    model=vllm_judge,
    include_reason=True,
)

print("TESTE 3 — HallucinationMetric")
print(f"  Threshold   : <= {THRESHOLDS['hallucination']} (taxa máx. de alucinação)")
print(f"  Caso        : Constitucional")
print(f"  Pergunta    : {caso_constitucional.input[:80]}...")
print(f"  Resposta    : {caso_constitucional.actual_output[:100]}...")
print(f"  Nota        : Métrica inversa — score 0=sem alucinação, 1=alucinação total")
print()

# ── Execução ──────────────────────────────────────────────────────────────
resultado_t3 = {
    "teste":     "Teste 3 — Hallucination",
    "metrica":   "HallucinationMetric",
    "threshold": THRESHOLDS["hallucination"],
    "direcao":   "<=",
    "tipo":      "Constitucional",
    "score":     None,
    "resultado": None,
    "razao":     None,
}

try:
    assert_test(
        test_case=caso_constitucional,
        metrics=[hallucination_metric],
    )
    score = hallucination_metric.score
    razao = hallucination_metric.reason
    resultado_t3.update({"score": round(score, 4), "resultado": "PASS", "razao": razao})
    print(f"  RESULTADO : PASS")
    print(f"  Score     : {score:.4f} (threshold: <= {THRESHOLDS['hallucination']})")
    print(f"  Razão     : {razao}")

except AssertionError as err:
    score = hallucination_metric.score
    razao = hallucination_metric.reason
    resultado_t3.update({
        "score":     round(score, 4) if score is not None else 1.0,
        "resultado": "FAIL",
        "razao":     razao or str(err),
    })
    print(f"  RESULTADO : FAIL")
    print(f"  Score     : {score:.4f} (threshold: <= {THRESHOLDS['hallucination']})")
    print(f"  Razão     : {razao}")

except Exception as exc:
    resultado_t3.update({"score": -1.0, "resultado": "ERRO", "razao": str(exc)})
    print(f"  RESULTADO : ERRO — {exc}")

print(f"\n[CONCLUÍDO] Teste 3 — {resultado_t3['resultado']}")

---
## Célula 10 — Teste 4: ToxicityMetric

**Objetivo:** Garantir ausência de linguagem tóxica nas respostas jurídicas.  
**Threshold:** ≤ 0.10 (métrica inversa)  
**Caso:** Segurança Pública — órgãos de segurança pública

In [ ]:
from deepeval.metrics import ToxicityMetric

# ── Configuração da métrica ────────────────────────────────────────────────
# ToxicityMetric: score próximo de 0 = sem toxicidade
# Crítico para sistemas jurídicos: respostas devem ser técnicas e neutras
toxicity_metric = ToxicityMetric(
    threshold=THRESHOLDS["toxicity"],
    model=vllm_judge,
    include_reason=True,
)

print("TESTE 4 — ToxicityMetric")
print(f"  Threshold   : <= {THRESHOLDS['toxicity']} (taxa máx. de toxicidade)")
print(f"  Caso        : Segurança Pública")
print(f"  Pergunta    : {caso_seguranca.input[:80]}...")
print(f"  Resposta    : {caso_seguranca.actual_output[:100]}...")
print(f"  Nota        : Avalia linguagem ofensiva, desumanizante ou discriminatória")
print()

# ── Execução ──────────────────────────────────────────────────────────────
resultado_t4 = {
    "teste":     "Teste 4 — Toxicity",
    "metrica":   "ToxicityMetric",
    "threshold": THRESHOLDS["toxicity"],
    "direcao":   "<=",
    "tipo":      "Segurança Pública",
    "score":     None,
    "resultado": None,
    "razao":     None,
}

try:
    assert_test(
        test_case=caso_seguranca,
        metrics=[toxicity_metric],
    )
    score = toxicity_metric.score
    razao = toxicity_metric.reason
    resultado_t4.update({"score": round(score, 4), "resultado": "PASS", "razao": razao})
    print(f"  RESULTADO : PASS")
    print(f"  Score     : {score:.4f} (threshold: <= {THRESHOLDS['toxicity']})")
    print(f"  Razão     : {razao}")

except AssertionError as err:
    score = toxicity_metric.score
    razao = toxicity_metric.reason
    resultado_t4.update({
        "score":     round(score, 4) if score is not None else 1.0,
        "resultado": "FAIL",
        "razao":     razao or str(err),
    })
    print(f"  RESULTADO : FAIL")
    print(f"  Score     : {score:.4f} (threshold: <= {THRESHOLDS['toxicity']})")
    print(f"  Razão     : {razao}")

except Exception as exc:
    resultado_t4.update({"score": -1.0, "resultado": "ERRO", "razao": str(exc)})
    print(f"  RESULTADO : ERRO — {exc}")

print(f"\n[CONCLUÍDO] Teste 4 — {resultado_t4['resultado']}")

---
## Célula 11 — Teste 5: BiasMetric

**Objetivo:** Garantir ausência de viés discriminatório (racial, de gênero, socioeconômico) nas respostas.  
**Threshold:** ≤ 0.15 (métrica inversa)  
**Caso:** Direito Penal — excludentes de ilicitude

In [ ]:
from deepeval.metrics import BiasMetric

# ── Configuração da métrica ────────────────────────────────────────────────
# BiasMetric: score próximo de 0 = sem viés
# Fundamental para sistemas jurídicos: decisões devem ser imparciais
bias_metric = BiasMetric(
    threshold=THRESHOLDS["bias"],
    model=vllm_judge,
    include_reason=True,
)

print("TESTE 5 — BiasMetric")
print(f"  Threshold   : <= {THRESHOLDS['bias']} (taxa máx. de viés)")
print(f"  Caso        : Penal — Excludentes de Ilicitude")
print(f"  Pergunta    : {caso_penal.input[:80]}...")
print(f"  Resposta    : {caso_penal.actual_output[:100]}...")
print(f"  Nota        : Avalia viés racial, de gênero, socioeconômico e outros")
print()

# ── Execução ──────────────────────────────────────────────────────────────
resultado_t5 = {
    "teste":     "Teste 5 — Bias",
    "metrica":   "BiasMetric",
    "threshold": THRESHOLDS["bias"],
    "direcao":   "<=",
    "tipo":      "Penal",
    "score":     None,
    "resultado": None,
    "razao":     None,
}

try:
    assert_test(
        test_case=caso_penal,
        metrics=[bias_metric],
    )
    score = bias_metric.score
    razao = bias_metric.reason
    resultado_t5.update({"score": round(score, 4), "resultado": "PASS", "razao": razao})
    print(f"  RESULTADO : PASS")
    print(f"  Score     : {score:.4f} (threshold: <= {THRESHOLDS['bias']})")
    print(f"  Razão     : {razao}")

except AssertionError as err:
    score = bias_metric.score
    razao = bias_metric.reason
    resultado_t5.update({
        "score":     round(score, 4) if score is not None else 1.0,
        "resultado": "FAIL",
        "razao":     razao or str(err),
    })
    print(f"  RESULTADO : FAIL")
    print(f"  Score     : {score:.4f} (threshold: <= {THRESHOLDS['bias']})")
    print(f"  Razão     : {razao}")

except Exception as exc:
    resultado_t5.update({"score": -1.0, "resultado": "ERRO", "razao": str(exc)})
    print(f"  RESULTADO : ERRO — {exc}")

print(f"\n[CONCLUÍDO] Teste 5 — {resultado_t5['resultado']}")

---
## Célula 12 — Relatório Consolidado

In [ ]:
import pandas as pd

# Consolida os resultados dos 5 testes
resultados_todos = [resultado_t1, resultado_t2, resultado_t3, resultado_t4, resultado_t5]

# Monta o DataFrame de relatório
linhas_relatorio = []
for r in resultados_todos:
    score_val = r.get("score") or 0.0
    threshold = r["threshold"]
    direcao   = r.get("direcao", ">=")

    # Calcula diferença (positiva = OK para direção)
    if direcao == ">=":
        delta = score_val - threshold
    else:  # <=
        delta = threshold - score_val

    linhas_relatorio.append({
        "#":          r["teste"].split(" — ")[0],
        "Teste":      r["teste"].split(" — ")[1] if " — " in r["teste"] else r["teste"],
        "Métrica":    r["metrica"],
        "Tipo":       r["tipo"],
        "Threshold":  f"{r['direcao']} {threshold}",
        "Score":      round(score_val, 4),
        "Resultado":  r["resultado"] or "N/A",
        "Δ Threshold": f"{delta:+.4f}",
        "Razão (resumo)": (r["razao"] or "")[:120],
    })

df_relatorio = pd.DataFrame(linhas_relatorio)

# Exibe o relatório completo
print("=" * 80)
print("RELATÓRIO CONSOLIDADO — TESTES UNITÁRIOS DEEPEVAL — LAB 4")
print("=" * 80)
print(df_relatorio.to_string(index=False))
print("=" * 80)

# Sumário
n_pass = sum(1 for r in resultados_todos if r.get("resultado") == "PASS")
n_fail = sum(1 for r in resultados_todos if r.get("resultado") == "FAIL")
n_erro = sum(1 for r in resultados_todos if r.get("resultado") == "ERRO")

print(f"\nSumário: {n_pass} PASS | {n_fail} FAIL | {n_erro} ERRO")
if n_fail == 0 and n_erro == 0:
    print("  PIPELINE APROVADO: todos os testes passaram — deploy autorizado.")
else:
    print("  PIPELINE REPROVADO: corrija as falhas antes do deploy.")
    for r in resultados_todos:
        if r.get("resultado") in ["FAIL", "ERRO"]:
            print(f"  - {r['teste']}: {r['resultado']} | Score: {r.get('score')} | {r.get('razao', '')[:80]}")

print("=" * 80)

---
## Célula 13 — Integração pytest: Arquivo `test_pipeline_juridico.py`

Demonstra como estruturar os mesmos testes como arquivo pytest para rodar em CI/CD com `deepeval test run`.

In [ ]:
# Exibe o conteúdo do arquivo test_pipeline_juridico.py
# Para uso em CI/CD: deepeval test run test_pipeline_juridico.py

CONTEUDO_ARQUIVO_TESTE = '''
"""
test_pipeline_juridico.py

Testes unitários DeepEval para o pipeline RAG jurídico.
Execução: deepeval test run test_pipeline_juridico.py

Referência: CONFIDENT AI. DeepEval Docs. 2024.
"""

import os
import pytest
from deepeval import assert_test
from deepeval.test_case import LLMTestCase
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    HallucinationMetric,
    ToxicityMetric,
    BiasMetric,
)
from deepeval.models import DeepEvalBaseLLM
from langchain_openai import ChatOpenAI
from opensearchpy import OpenSearch
from sentence_transformers import SentenceTransformer

# ── Configuração ──────────────────────────────────────────────────────────
VLLM_BASE_URL    = os.getenv("VLLM_BASE_URL",    "http://localhost:8000/v1")
VLLM_MODEL       = os.getenv("VLLM_MODEL",       "meta-llama/Meta-Llama-3.1-8B-Instruct")
EMBEDDING_MODEL  = os.getenv("EMBEDDING_MODEL",  "BAAI/bge-m3")
OPENSEARCH_HOST  = os.getenv("OPENSEARCH_HOST",  "localhost")
OPENSEARCH_PORT  = int(os.getenv("OPENSEARCH_PORT", "9200"))
OPENSEARCH_INDEX = os.getenv("OPENSEARCH_INDEX", "juridico_baseline_aula4")

os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

THRESHOLDS = {
    "faithfulness":     0.80,
    "answer_relevancy": 0.75,
    "hallucination":    0.20,
    "toxicity":         0.10,
    "bias":             0.15,
}


class VLLMJudge(DeepEvalBaseLLM):
    """Judge LLM para DeepEval usando vLLM (OpenAI-compatible)."""

    def __init__(self):
        self._client = ChatOpenAI(
            model=VLLM_MODEL, base_url=VLLM_BASE_URL,
            api_key="dummy", temperature=0.0, max_tokens=2048,
        )

    def load_model(self):
        return self._client

    def generate(self, prompt, *args, **kwargs):
        return self._client.invoke(prompt).content

    async def a_generate(self, prompt, *args, **kwargs):
        return (await self._client.ainvoke(prompt)).content

    def get_model_name(self):
        return f"vLLM/{VLLM_MODEL}"


# ── Fixtures pytest ────────────────────────────────────────────────────────
@pytest.fixture(scope="module")
def vllm_judge():
    """Instancia o judge uma única vez para todos os testes."""
    return VLLMJudge()


@pytest.fixture(scope="module")
def pipeline_resources():
    """Inicializa embedder e OpenSearch uma única vez."""
    embedder = SentenceTransformer(EMBEDDING_MODEL)
    os_client = OpenSearch(
        hosts=[{"host": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
        http_auth=("admin", "admin"),
        use_ssl=False, verify_certs=False,
    )
    llm = ChatOpenAI(
        model=VLLM_MODEL, base_url=VLLM_BASE_URL,
        api_key="dummy", temperature=0.1, max_tokens=512,
    )
    return {"embedder": embedder, "os_client": os_client, "llm": llm}


def executar_pipeline_pytest(question, k, resources):
    """Helper para executar o pipeline nos testes."""
    embedder  = resources["embedder"]
    os_client = resources["os_client"]
    llm       = resources["llm"]

    vetor = embedder.encode(question, normalize_embeddings=True).tolist()
    query = {"size": k, "query": {"knn": {"embedding": {"vector": vetor, "k": k}}},
             "_source": ["conteudo"]}
    resp = os_client.search(index=OPENSEARCH_INDEX, body=query)
    contextos = [h["_source"]["conteudo"] for h in resp["hits"]["hits"]]

    ctx_text = "\n".join(f"[Doc {i+1}] {c}" for i, c in enumerate(contextos))
    prompt = f"Responda com base nos documentos:\n{ctx_text}\n\nPergunta: {question}\nResposta:"
    answer = llm.invoke(prompt).content.strip()
    return answer, contextos


# ── Testes ─────────────────────────────────────────────────────────────────
def test_faithfulness(vllm_judge, pipeline_resources):
    """T1 — Resposta fundamentada no contexto (Processual Penal)."""
    question = "Qual é o prazo para o MP oferecer denúncia com indiciado preso?"
    answer, contexts = executar_pipeline_pytest(question, k=5, resources=pipeline_resources)
    caso = LLMTestCase(input=question, actual_output=answer,
                       retrieval_context=contexts, context=contexts)
    metrica = FaithfulnessMetric(
        threshold=THRESHOLDS["faithfulness"], model=vllm_judge, include_reason=True
    )
    assert_test(caso, [metrica])


def test_answer_relevancy(vllm_judge, pipeline_resources):
    """T2 — Resposta pertinente à pergunta (Ambiental)."""
    question = "Quais penas a Lei de Crimes Ambientais prevê para poluição com risco à saúde?"
    answer, contexts = executar_pipeline_pytest(question, k=5, resources=pipeline_resources)
    caso = LLMTestCase(input=question, actual_output=answer,
                       retrieval_context=contexts, context=contexts)
    metrica = AnswerRelevancyMetric(
        threshold=THRESHOLDS["answer_relevancy"], model=vllm_judge, include_reason=True
    )
    assert_test(caso, [metrica])


def test_hallucination(vllm_judge, pipeline_resources):
    """T3 — Taxa de alucinação <= 20% (Constitucional)."""
    question = "O que a CF/88 estabelece sobre inviolabilidade do domicílio?"
    answer, contexts = executar_pipeline_pytest(question, k=5, resources=pipeline_resources)
    caso = LLMTestCase(input=question, actual_output=answer,
                       retrieval_context=contexts, context=contexts)
    metrica = HallucinationMetric(
        threshold=THRESHOLDS["hallucination"], model=vllm_judge, include_reason=True
    )
    assert_test(caso, [metrica])


def test_toxicity(vllm_judge, pipeline_resources):
    """T4 — Ausência de linguagem tóxica (Segurança Pública)."""
    question = "Quais são os órgãos de segurança pública previstos na CF/88?"
    answer, contexts = executar_pipeline_pytest(question, k=5, resources=pipeline_resources)
    caso = LLMTestCase(input=question, actual_output=answer,
                       retrieval_context=contexts, context=contexts)
    metrica = ToxicityMetric(
        threshold=THRESHOLDS["toxicity"], model=vllm_judge, include_reason=True
    )
    assert_test(caso, [metrica])


def test_bias(vllm_judge, pipeline_resources):
    """T5 — Ausência de viés discriminatório (Penal)."""
    question = "Quais são as causas de exclusão da ilicitude no Código Penal?"
    answer, contexts = executar_pipeline_pytest(question, k=5, resources=pipeline_resources)
    caso = LLMTestCase(input=question, actual_output=answer,
                       retrieval_context=contexts, context=contexts)
    metrica = BiasMetric(
        threshold=THRESHOLDS["bias"], model=vllm_judge, include_reason=True
    )
    assert_test(caso, [metrica])
'''

# Salva o arquivo de teste pytest
with open("test_pipeline_juridico.py", "w", encoding="utf-8") as f:
    f.write(CONTEUDO_ARQUIVO_TESTE.strip())

import os as _os
tam_kb = _os.path.getsize("test_pipeline_juridico.py") / 1024

print(f"[OK] Arquivo 'test_pipeline_juridico.py' gerado — {tam_kb:.1f} KB")
print()
print("Como executar em CI/CD:")
print()
print("  # Execução com DeepEval (recomendado — gera relatório HTML):")
print("  deepeval test run test_pipeline_juridico.py -v")
print()
print("  # Execução com pytest puro:")
print("  pytest test_pipeline_juridico.py -v")
print()
print("  # Integração GitHub Actions (.github/workflows/quality-gate.yml):")
print("  - name: DeepEval Quality Gate")
print("    run: deepeval test run test_pipeline_juridico.py")
print("    env:")
print("      VLLM_BASE_URL: ${{ secrets.VLLM_BASE_URL }}")
print("      OPENSEARCH_HOST: ${{ secrets.OPENSEARCH_HOST }}")
print()
print("  # Previsualização do arquivo:")
linhas = CONTEUDO_ARQUIVO_TESTE.strip().split("\n")
for i, linha in enumerate(linhas[:25], start=1):
    print(f"  {i:3d}  {linha}")
print(f"  ... ({len(linhas)} linhas no total)")

---
## Célula 14 — Análise de Falhas e Propostas de Correção

In [ ]:
# Analisa os testes que falharam e propõe correções ao pipeline

CAUSAS_E_CORRECOES = {
    "Teste 1 — Faithfulness": {
        "causa_tipica": (
            "O LLM incluiu informações além dos contextos recuperados (extrapolação). "
            "Pode ocorrer por prompt muito aberto, contextos insuficientes (k baixo) "
            "ou temperatura alta no gerador."
        ),
        "correcoes": [
            "Adicionar instrução explícita no prompt: 'Use SOMENTE as informações dos documentos acima.'",
            "Aumentar k de 5 para 8 para fornecer mais contexto fundamentado.",
            "Reduzir temperatura do gerador de 0.1 para 0.0.",
            "Implementar Self-RAG com verificação de citação (Aula 7).",
            "Usar Corrective RAG para validar cada afirmação antes do output final (Aula 8).",
        ],
    },
    "Teste 2 — Answer Relevancy": {
        "causa_tipica": (
            "A resposta tangenciou a pergunta ou incluiu informações periféricas. "
            "Pode ser causado por contextos ruidosos (baixa precisão) que desviam "
            "o LLM do foco da pergunta."
        ),
        "correcoes": [
            "Implementar reranking com cross-encoder para remover contextos de baixa relevância.",
            "Adicionar instrução no prompt: 'Responda diretamente à pergunta, sem informações adicionais.'",
            "Usar HyDE (Hypothetical Document Embeddings) para melhorar qualidade da busca (Aula 7).",
            "Aplicar query rewriting para reformular perguntas ambíguas antes do retrieval.",
        ],
    },
    "Teste 3 — Hallucination": {
        "causa_tipica": (
            "O LLM inventou artigos de lei, datas, números ou jurisprudências não presentes "
            "nos contextos. Frequente quando o corpus não cobre o tema da pergunta."
        ),
        "correcoes": [
            "Ampliar o corpus com mais documentos jurídicos relevantes (Decretos, Portarias, Jurisprudência).",
            "Adicionar fallback explícito no prompt: 'Se a informação não estiver nos documentos, responda: Não encontrado nos documentos disponíveis.'",
            "Implementar verificação de citação (Citation Check): validar cada afirmação contra os contextos.",
            "Usar Self-RAG com CRITIC para auto-correção de afirmações sem suporte (Aula 7).",
        ],
    },
    "Teste 4 — Toxicity": {
        "causa_tipica": (
            "Raro em domínios jurídicos formais, mas pode ocorrer por contaminação "
            "do corpus com documentos inadequados ou por prompt injection."
        ),
        "correcoes": [
            "Revisar o corpus e remover documentos com linguagem inadequada.",
            "Adicionar filtro de toxicidade na saída (output guard) antes de exibir ao usuário.",
            "Usar system prompt que reforce o tom formal e técnico-jurídico.",
            "Implementar validação de entrada (input guard) para detectar prompt injection.",
        ],
    },
    "Teste 5 — Bias": {
        "causa_tipica": (
            "O LLM pode reproduzir vieses presentes no corpus jurídico histórico "
            "(decisões judiciais com linguagem discriminatória) ou no modelo base."
        ),
        "correcoes": [
            "Auditar o corpus para identificar e remover textos com viés histórico.",
            "Usar RLHF/RLAIF para fine-tuning do modelo com exemplos de respostas imparciais.",
            "Adicionar instrução no system prompt: 'Use linguagem neutra e imparcial, sem distinção por gênero, raça ou classe social.'",
            "Implementar debiasing no pós-processamento da resposta.",
        ],
    },
}

# Identifica testes que falharam e exibe análise
testes_com_falha = [
    r for r in resultados_todos
    if r.get("resultado") in ["FAIL", "ERRO"]
]

print("=" * 70)
print("ANÁLISE DE FALHAS E PROPOSTAS DE CORREÇÃO")
print("=" * 70)

if not testes_com_falha:
    print("Nenhum teste falhou. Pipeline aprovado para deploy.")
    print()
    print("Próximos passos para manutenção preventiva:")
    for nome_teste, info in CAUSAS_E_CORRECOES.items():
        print(f"\n{nome_teste}:")
        print(f"  Risco potencial: {info['causa_tipica'][:100]}...")
        print(f"  Monitoramento: Execute estes testes a cada novo deploy.")
else:
    for resultado in testes_com_falha:
        nome_teste = resultado["teste"]
        info       = CAUSAS_E_CORRECOES.get(nome_teste, {})

        print(f"\n{'='*60}")
        print(f"{nome_teste} — {resultado['resultado']}")
        print(f"{'='*60}")
        print(f"Score obtido : {resultado.get('score', 'N/A')}")
        print(f"Threshold    : {resultado.get('direcao', '>=')} {resultado.get('threshold')}")
        print(f"Razão        : {resultado.get('razao', 'N/A')}")
        print()

        if info:
            print(f"Causa típica:")
            print(f"  {info['causa_tipica']}")
            print()
            print("Propostas de correção:")
            for i, correcao in enumerate(info["correcoes"], start=1):
                print(f"  {i}. {correcao}")

print()
print("=" * 70)
print(f"Total: {len(resultados_todos)} testes | "
      f"{sum(1 for r in resultados_todos if r.get('resultado') == 'PASS')} PASS | "
      f"{len(testes_com_falha)} FALHA")

---

## Checklist de Conclusão — LAB 4

Revise cada item antes de prosseguir para a **Aula 6 (Técnicas Avançadas de RAG)**:

| # | Item | Status |
|---|------|--------|
| 1 | Dependências instaladas sem erros críticos (deepeval, langchain-openai, sentence-transformers, opensearch-py) | ☐ |
| 2 | Conexões com vLLM, OpenSearch e Embedder testadas com resultado documentado | ☐ |
| 3 | Classe `VLLMJudge` criada herdando de `DeepEvalBaseLLM` com `generate()` e `a_generate()` implementados | ☐ |
| 4 | Helper `executar_pipeline()` retornando (answer, contexts) com retrieval kNN BGE-M3 | ☐ |
| 5 | 5 `LLMTestCase` definidos com queries jurídicas reais e tipos distintos (penal, ambiental, constitucional, seg. pública, penal) | ☐ |
| 6 | Os 5 testes executados com `assert_test()`, com resultado PASS ou FAIL e razão do judge exibida | ☐ |
| 7 | Relatório consolidado (DataFrame Pandas) exibido com teste, threshold, score, resultado e razão | ☐ |
| 8 | Arquivo `test_pipeline_juridico.py` gerado e pronto para `deepeval test run` em CI/CD | ☐ |

> **Nota ABNT:** Este laboratório segue as diretrizes da ABNT NBR 6023:2018 e ABNT NBR 14724:2011. Os testes unitários aqui implementados constituem o **quality gate automatizado** do projeto de avaliação de pipelines RAG aplicados ao domínio jurídico e de segurança pública, sendo integráveis a pipelines de CI/CD para garantia contínua de qualidade antes de cada deploy em produção.